# Baseline feature extraction

Notebook ini mengekstrak fitur operasional gegenpressing untuk seluruh queue
anotasi. Fitur hanya berasal dari tracking pada dua detik setelah loss. Outcome
quick regain dan label manual tidak digunakan selama ekstraksi, sehingga tidak
terjadi target leakage.

Metodologi tidak berubah: fitur mencakup jarak, approach rate, jumlah presser,
compactness, dan durasi tekanan. Tabel ini disiapkan untuk baseline setelah
anotasi penuh selesai.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.gegenpressing_features import (
    BASELINE_FEATURE_COLUMNS,
    audit_baseline_feature_table,
    build_baseline_feature_table,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'sportec_idsse'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ANNOTATION_PATH = PROCESSED_DIR / 'gegenpressing_annotation_manifest.csv'
FEATURE_PATH = PROCESSED_DIR / 'gegenpressing_baseline_features.csv'

In [2]:
annotations = pd.read_csv(ANNOTATION_PATH)
features = build_baseline_feature_table(annotations, DATA_DIR)
features.to_csv(FEATURE_PATH, index=False)

print('Candidates:', len(annotations))
print('Feature rows:', len(features))
print('Feature columns:', len(BASELINE_FEATURE_COLUMNS))
print('Saved:', FEATURE_PATH)

Candidates: 264
Feature rows: 264
Feature columns: 12
Saved: C:\VAEP-Track\data\processed\gegenpressing_baseline_features.csv


In [3]:
checks = audit_baseline_feature_table(features, annotations)
display(checks.to_frame())
assert checks.all(), 'Audit baseline feature table gagal.'

missing = features[BASELINE_FEATURE_COLUMNS].isna().sum()
print('Missing values by feature:')
display(missing[missing.gt(0)].rename('missing').to_frame())
print('Feature summary:')
display(features[BASELINE_FEATURE_COLUMNS].describe().T)

,passed
candidate_id_unique,True
candidate_ids_complete,True
feature_columns_complete,True
no_infinite_numeric_values,True
outcome_not_in_features,True
gegenpressing_label_not_in_features,True


Missing values by feature:


,missing
mean_local_compactness_m,14


Feature summary:


,count,mean,std,min,25%,50%,75%,max
max_total_pressers,264.0,2.541667,1.416285,0.000000,2.000000,2.000000,3.000000,10.000000
max_direct_pressers,264.0,1.587121,0.967133,0.000000,1.000000,1.000000,2.000000,6.000000
max_approaching_pressers,264.0,2.090909,1.236053,0.000000,1.000000,2.000000,3.000000,9.000000
pressure_duration_s,264.0,1.529697,0.670901,0.000000,1.000000,2.040000,2.040000,2.040000
collective_pressure_duration_s,264.0,0.771061,0.730647,0.000000,0.080000,0.520000,1.330000,2.040000
minimum_losing_distance_m,264.0,1.892637,3.048655,0.010000,0.402616,0.870345,1.893418,22.518615
maximum_approach_rate_mps,264.0,12.378158,6.111710,-0.611875,8.025046,11.677993,15.671675,37.854194
max_losing_players_within_10m,264.0,3.238636,1.699884,0.000000,2.000000,3.000000,4.000000,10.000000
mean_local_compactness_m,250.0,6.850228,1.347256,3.580480,5.942255,6.864089,7.904082,10.166345
team_stretch_change_m,264.0,-0.341537,1.195660,-4.709183,-1.034941,-0.344985,0.559780,2.544110


In [4]:
labels = annotations[[
    'candidate_id',
    'is_gegenpressing',
    'annotation_confidence',
]].copy()
audit = features.merge(labels, on='candidate_id', how='left', validate='one_to_one')
completed = audit[audit['is_gegenpressing'].notna()].copy()
usable = completed[completed['is_gegenpressing'].isin([0, 1])].copy()

print('Completed annotations:', len(completed))
print('Usable binary labels:', len(usable))
print('Remaining annotations:', len(audit) - len(completed))
if len(usable):
    display(
        usable['is_gegenpressing']
        .astype(int)
        .value_counts()
        .sort_index()
        .rename('count')
        .to_frame()
    )
print('Status: features ready; final baseline waits for full annotation.')

Completed annotations: 28
Usable binary labels: 26
Remaining annotations: 236


,count
is_gegenpressing,
0,5
1,21


Status: features ready; final baseline waits for full annotation.
